In [1]:
import pandas as pd
import pickle
from utils import model as mod

In [2]:
df = pd.read_pickle('../data/03_df_data')
segs = pd.read_pickle('../data/02_df_seg_race')

with open("../data/02_dic_ref_groups.pkl", "rb") as file:
    ref_groups = pickle.load(file)

In [3]:
df.head()

,loan_type,loan_purpose,preapproval,construction_method,loan_amount,action_taken,state_code,county_code,census_tract,applicant_ethnicity_1,...,intro_rate_period,balloon_payment,interest_only_payment,property_value,manufactured_home_secured_property_type,manufactured_home_land_property_interest,total_units,reg_uw,denied,reg_price
1,Conventional,Home purchase,Not requested,Site built,1205000,Purchased,WA,53033.0,53033032318.0,Not Hispanic Latino,...,NaN,No balloon,Not interest only,1505000.0,Not applicable,Not applicable,1,0,0,0
2,Conventional,Home purchase,Not requested,Site built,925000,Purchased,WA,53011.0,53011040303.0,Info not provided,...,NaN,No balloon,Not interest only,1035000.0,Not applicable,Not applicable,1,0,0,0
5,Conventional,Home purchase,Not requested,Site built,905000,Purchased,WA,53011.0,53011040905.0,Not Hispanic Latino,...,NaN,No balloon,Not interest only,1135000.0,Not applicable,Not applicable,1,0,0,0
13,Conventional,Home purchase,Not requested,Site built,1505000,Purchased,CA,6065.0,6065045606.0,Not Hispanic Latino,...,NaN,No balloon,Not interest only,4295000.0,Not applicable,Not applicable,1,0,0,0
15,Conventional,Refinance,Not requested,Site built,1145000,Purchased,WA,53033.0,53033002700.0,Info not provided,...,84.0,No balloon,Not interest only,1725000.0,Not applicable,Not applicable,1,0,0,0


In [4]:
segs

,loan_type,loan_purpose,applicant_race_1,applied,event_count,event_rate,mi_rate,mu_rate,mx_rate,crit event,crit count
0,Conventional,Cash out refinance,Black African American,1607,218,0.135657,4.490,6.720161,9.74,True,True
1,Conventional,Cash out refinance,White,8725,1163,0.133295,2.875,6.756857,10.24,True,True
4,Conventional,Home purchase,Black African American,4562,458,0.100395,2.500,6.358342,9.24,True,True
5,Conventional,Home purchase,White,34364,2132,0.062042,1.000,6.356195,9.49,True,True
12,FHA insured,Home purchase,Black African American,883,124,0.140430,4.125,6.339181,7.75,True,True
13,FHA insured,Home purchase,White,2250,210,0.093333,3.750,6.295870,7.75,True,True


In [5]:
testing = segs.columns.tolist()[2]
testing

'applicant_race_1'

In [6]:
ref_groups

{'applicant_race_1': 'White',
 'applicant_sex': 'Male',
 'applicant_age_above_62': 'No'}

In [7]:
segs.loan_type.unique().tolist()

['Conventional', 'FHA insured']

In [8]:
drop_this = [
    'applicant_ethnicity_1',        
    'applicant_race_1',                  
    'applicant_sex', 
    'applicant_age_above_62'
]

In [9]:
# %%capture output

for type_i in segs.loan_type.unique().tolist():
    print(f'\nType: {type_i}')
    
    for purp_i in segs[segs['loan_type'] == type_i].loan_purpose.unique().tolist():
        print(f'\n Purpose: {purp_i}')

        group_list = segs[(segs['loan_type'] == type_i)&(segs['loan_purpose'] == purp_i)][testing].unique().tolist()
        group_list.remove(ref_groups[testing])
        for group_i in group_list:
            print(f'\n  PB: {group_i}')
            print(f'  Ref: {ref_groups[testing]}')
    
            """
            Underwriting
            """

            print('\nUnderwriting')
    
            df_tmp = df[
                (df.loan_type == type_i) &
                (df.loan_purpose == purp_i) &
                (df[testing].isin([ref_groups[testing]] +  [group_i])) &
                (df['reg_uw'] == 1)
            ]



            
            
            print(f'shape: {df_tmp.shape}')

            print(df_tmp.groupby(testing).agg(
                count=('denied','size'),
                event_count=('denied','sum')
            ))

            # make dummy for protected basis group

            df_tmp[group_i] = (df_tmp[testing] == group_i).astype(int)

            df_tmp.drop(columns = drop_this, inplace = True)
            
            # drop columns with single value. will end up being things like indicator for segment
            df_tmp = df_tmp.loc[:, df_tmp.nunique(dropna=False) > 1]
            

            print('')
            print(df_tmp[group_i].value_counts())
            
            """
            UW Model 0
            """

            print('\nModel 0')
            
            import statsmodels.api as sm
            
            X = sm.add_constant(df_tmp[group_i])
            y = df_tmp["denied"]
            
            
            model = sm.Logit(y, X)
            result = model.fit()
            
            print(result.summary())
    
    
            
            """
            UW Model 1
            """
            print('\nModel 1')

            result = mod.logistic_woe_run(df_tmp,group_i)
            
            
            """
            UW Model 2
            """
            print('\nModel 2')

            df_tmp_psa = mod.make_match_pair(df_tmp,'denied',group_i)
            
            result = mod.logistic_woe_run(df_tmp_psa,group_i)











            
            
    
            """
            Pricing
            """
            print('\nPricing')
                
            df_tmp = df[
                (df.loan_type == type_i) &
                (df.loan_purpose == purp_i) &
                (df[testing].isin([ref_groups[testing]] +  [group_i])) &
                (df['reg_price'] == 1)
            ]



            
            
            print(f'shape: {df_tmp.shape}')

            # make dummy for protected basis group

            df_tmp[group_i] = (df_tmp[testing] == group_i).astype(int)

            df_tmp.drop(columns = drop_this, inplace = True)
            
            # drop columns with single value. will end up being things like indicator for segment
            df_tmp = df_tmp.loc[:, df_tmp.nunique(dropna=False) > 1]
            

            print('')
            print(df_tmp[group_i].value_counts())
    
            
            """
            Pri Model 0
            """
            print('\nModel 0')
            
            import statsmodels.api as sm
            
            X = sm.add_constant(df_tmp[group_i])
            y = df_tmp["interest_rate"]
            
            
            model = sm.OLS(y, X)
            result = model.fit()
            
            print(result.summary())
    
    
    
    
            
            """
            Pri Model 1
            """
            print('\nModel 1')

            result = mod.ols_dummy_run(df_tmp,group_i)

            print(result.summary())

            
    
            """
            Pri Model 2
            """

            print('\nModel 2')

            #pair function needs to ignore something, gets excluded later bc singular value
            df_tmp['rnd'] = 999
            df_tmp['action_taken'] = 'junk'
            
            df_tmp_psa = mod.make_match_pair(df_tmp,'rnd',group_i)
            result = mod.ols_dummy_run(df_tmp_psa,group_i)

            print(result.summary())

            
            
    #         break # group_i

    #     break # purp_i
    # break # type_i




        


Type: Conventional

 Purpose: Cash out refinance

  PB: Black African American
  Ref: White

Underwriting
shape: (7003, 37)
                        count  event_count
applicant_race_1                          
Black African American   1044          218
White                    5959         1163

Black African American
0    5959
1    1044
Name: count, dtype: int64

Model 0
Optimization terminated successfully.
         Current function value: 0.496424
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                 denied   No. Observations:                 7003
Model:                          Logit   Df Residuals:                     7001
Method:                           MLE   Df Model:                            1
Date:                Tue, 15 Sep 2026   Pseudo R-squ.:               0.0001485
Time:                        20:40:50   Log-Likelihood:                -3476.5
converged:                       True   LL-Null:

In [11]:
print(
    'hello'
)

hello


In [1]:
df_tmp.info()

NameError: name 'df_tmp' is not defined

In [14]:
X.describe()

,const,Black African American
count,2872.0,2872.000000
mean,1.0,0.117688
std,0.0,0.322294
min,1.0,0.000000
25%,1.0,0.000000
50%,1.0,0.000000
75%,1.0,0.000000
max,1.0,1.000000


In [10]:
y.head()

106433    0
107460    0
126363    1
126375    0
126379    0
Name: denied, dtype: int64

In [15]:
df_tmp_psa.describe()

,loan_amount,income,combined_loan_to_value_ratio,loan_term,intro_rate_period,property_value,denied,Black African American
count,676.000000,670.000000,676.000000,676.000000,322.000000,6.760000e+02,676.000000,676.00000
mean,105399.408284,102.544776,47.351430,323.075444,1.257764,4.218787e+05,0.335799,0.50000
std,101984.691369,110.864197,23.457882,76.454074,4.625409,3.719246e+05,0.472619,0.50037
min,15000.000000,0.000000,3.000000,120.000000,1.000000,4.500000e+04,0.000000,0.00000
25%,45000.000000,46.000000,27.000000,360.000000,1.000000,2.050000e+05,0.000000,0.00000
50%,75000.000000,73.000000,48.000000,360.000000,1.000000,3.150000e+05,0.000000,0.50000
75%,125000.000000,118.000000,66.276250,360.000000,1.000000,5.350000e+05,1.000000,1.00000
max,965000.000000,1179.000000,152.754000,360.000000,84.000000,4.105000e+06,1.000000,1.00000


In [16]:
from utils import model as mod
import statsmodels.api as sm

# check separtion and remove those columns
flagged_variables = mod.find_separation_variables(
    df=df_tmp,
    target="denied",
    print_results=False,
    check_near_separation=True
)

In [17]:
flagged_variables

[]

In [18]:
df_tmp.drop(columns = flagged_variables, inplace = True)

In [19]:
df_tmp.shape

(2872, 14)

In [20]:
#############create woe
import toad

# Ensure object columns are clean strings
object_cols = df_tmp.select_dtypes(include=["object", "string"]).columns

for col in object_cols:
    df_tmp[col] = df_tmp[col].astype("string").fillna("__MISSING__")



# Create the combiner
combiner = toad.transform.Combiner()

# Fit bins 
combiner.fit(
    df_tmp,
    y='denied',
    method="chi",
    min_samples=0.05,
    empty_separate=True, #edit
    exclude= [group_i]   #['denied']
)

In [21]:
# Apply bin rules
df_tmp_bin = combiner.transform(df_tmp)

# convert bins to woe 
woe_transformer = toad.transform.WOETransformer()

df_tmp_woe = woe_transformer.fit_transform(
    df_tmp_bin,
    df_tmp_bin['denied'],
    exclude=['denied',group_i]
)

In [22]:
####### assign x and y data
y = df_tmp_woe["denied"]

# drop columns with single value. will end up being things like indicator for segment
df_tmp_woe = df_tmp_woe.loc[:, df_tmp_woe.nunique(dropna=False) > 1]

# reorder columns
col_order = df_tmp_woe.columns.tolist()
col_order.remove(group_i)
df_tmp_woe = df_tmp_woe[[group_i] + col_order]

X = sm.add_constant(df_tmp_woe.drop(columns='denied'))

In [24]:
model = sm.Logit(y, X)
result = model.fit()

print(result.summary())

# return result

         Current function value: 0.558696
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                 denied   No. Observations:                 2872
Model:                          Logit   Df Residuals:                     2859
Method:                           MLE   Df Model:                           12
Date:                Tue, 15 Sep 2026   Pseudo R-squ.:                 0.05399
Time:                        14:41:16   Log-Likelihood:                -1604.6
converged:                      False   LL-Null:                       -1696.2
Covariance Type:            nonrobust   LLR p-value:                 9.566e-33
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                             -1.0306      0.766     -1.345      0.179      -2.532       0.471
Black

/home/jovyan/work/proj envs/fair-lend/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
